# 04 — Train Unet2D L1 on random-window export

Trains adv-NO on **2013–2022** data exported by `Shalow/01_export_random_window.ipynb`. Rollout eval in [05_Sh_evaluate_magic_window_rollout.ipynb](05_Sh_evaluate_magic_window_rollout.ipynb).

**All code, checkpoints, logs, and rollout outputs run from and save under `Shalow/`.**

**Change only `EXPORT_KEY`** in the setup cell below (same model/settings for all datasets). Run name and output folder are derived automatically.

| Item | Value |
|---|---|
| Data root | `Shalow/data/{EXPORT_KEY}/` |
| Train pool | `sample_13_22_uv_dt360` (noon-only, **K=2 crops/anchor** in export; train with **`TRAIN_CROPS_PER_ANCHOR = 1`**) |
| Training crops | **Dynamic Zarr** (`DYNAMIC_RANDOM_CROP = True`) — new random crop each epoch; **parallel CPU workers** + **single GPU `cuda:7`** |
| Alt mode | `DYNAMIC_RANDOM_CROP = False` → fixed `_sims.npy` in RAM (fast; tag `fixwin_...`) |
| Val split | **Random** 70/30 (`VAL_SPLIT_MODE = "random"`, seed=42) |
| Loss | **L1 only** (`LOSS_MODE = "l1"`) |
| Generator | **`gUnet`** with **`UNET_DROPOUT = 0.1`** |
| Run dir | `..._drop0.1_dynwin_l1random_k1` (dynamic, 1 crop/anchor, random val) |

**Re-export?** No — existing K=2 export is fine. `TRAIN_CROPS_PER_ANCHOR = 1` keeps only `crop_slot == 0` rows (~3652 anchors). Dynamic mode re-crops from Zarr each epoch anyway.

**Fresh run:** `RESUME = False`, `PROTECT_EXISTING_RUN = False` — overwrites `{RUN_DIR}` if it exists.

**Run order:** setup → export check → dataset stats → train config → preview → train → loss curves.

**Saved for comparison:** `{RUN_DIR}/dataset_stats.json`, `{RUN_DIR}/best_model.pth`, `norm_stats.pt`, plus `Shalow/Models/adv_no/comparison_index.json`


In [1]:
import json
import subprocess
import os
import sys
from dataclasses import asdict
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
SHALOW_DIR = NOTEBOOK_DIR if NOTEBOOK_DIR.name == "Shalow" else NOTEBOOK_DIR / "Shalow"
ROOT = SHALOW_DIR.parent
os.chdir(SHALOW_DIR)
for _p in (ROOT, SHALOW_DIR):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import importlib
import run_registry as _run_registry
importlib.reload(_run_registry)

from window_paths import pool_prefix, test_prefix, window_data_dir
from run_registry import (
    RUN_TAG,
    collect_dataset_stats,
    run_dir,
    run_name,
    weight_decay_run_tag,
    generator_run_tag,
    dropout_run_tag,
    save_dataset_stats,
    update_comparison_index,
)

DATA_DIR = SHALOW_DIR / "data"
MODELS_DIR = SHALOW_DIR / "Models" / "adv_no"
LOG_DIR = SHALOW_DIR / "logs"
PNG_DIR = SHALOW_DIR / "PNG"
TEST_ROLLOUT_DIR_ROOT = SHALOW_DIR / "Test&Rollout"
for _d in (MODELS_DIR, LOG_DIR, PNG_DIR, TEST_ROLLOUT_DIR_ROOT):
    _d.mkdir(parents=True, exist_ok=True)

import src.train.fix_window_train as _fwt
import src.train.fix_window_viz as _fwv
import src.train.losses as _losses
import train.adv_no_train as _adv
importlib.reload(_losses)
importlib.reload(_fwt)
importlib.reload(_fwv)
importlib.reload(_adv)

from train.adv_no_train import AdvNOTrainConfig, run_adv_no_training
from src.train.fix_window_train import (
    load_merged_training_arrays,
    load_training_arrays,
    resolve_train_val_indices,
    prepare_train_val_tensors,
)


def load_notebook_training_arrays(cfg):
    """Load arrays for preview/config cells; mirrors run_adv_no_training."""
    if cfg.merge_val_export_prefix and not cfg.val_data_prefix:
        return load_merged_training_arrays(cfg), None
    data, mask, day, hour, crop_h, crop_w = load_training_arrays(cfg)
    if cfg.val_data_prefix:
        val_tensors = load_training_arrays(cfg, data_prefix=cfg.val_data_prefix)
        return (data, mask, day, hour, crop_h, crop_w), val_tensors
    return (data, mask, day, hour, crop_h, crop_w), None

# --- ONLY CHANGE THIS when switching datasets ---
EXPORT_KEY = "random_win_128"
# EXPORT_KEY = "legacy_smallfixwin_45deg"  # fixed window in Magic_window/data

DT = 360
DATA_PREFIX = pool_prefix(DATA_DIR, EXPORT_KEY, dt=DT)
TEST_DATA_PREFIX = test_prefix(DATA_DIR, EXPORT_KEY, dt=DT)
MAX_SAMPLES = None  # use all exported sequences

MAX_MASK_FRAC = 0.05  # must match export filter

VAL_SPLIT = 0.3  # 30% validation, 70% train
VAL_SPLIT_MODE = "random"  # "chronological" | "random"
VAL_SPLIT_SEED = 42

# --- fixed vs dynamic training (must match notebook 05) ---
DYNAMIC_RANDOM_CROP = True  # True: Zarr re-crop each epoch; False: fixed export in RAM
RANDOM_CROP_SEED = 42
TRAIN_CROPS_PER_ANCHOR = 1  # 1 = one export row per noon anchor (crop_slot 0); no re-export needed
# TRAIN_CROPS_PER_ANCHOR = 2  # both export crops per anchor (~7304 rows)
_val_suffix = "chrono" if VAL_SPLIT_MODE == "chronological" else VAL_SPLIT_MODE
_crop_prefix = "dynwin" if DYNAMIC_RANDOM_CROP else "fixwin"
_crops_tag = f"_k{int(TRAIN_CROPS_PER_ANCHOR)}" if int(TRAIN_CROPS_PER_ANCHOR) != 2 else ""
EXPERIMENT_TAG = f"{_crop_prefix}_l1{_val_suffix}{_crops_tag}"

# --- parallel throughput (dynamic Zarr crop is CPU-bound) ---
_export_cfg_path = Path(f"{DATA_PREFIX}_export_config.json")
_zarr_is_s3 = False
if _export_cfg_path.exists():
    _zarr_is_s3 = str(json.loads(_export_cfg_path.read_text()).get("zarr_path", "")).startswith("s3://")
_cpu_count = os.cpu_count() or 8
if DYNAMIC_RANDOM_CROP:
    # S3 Zarr: each worker opens its own client; cap workers to limit connection storms
    NUM_WORKERS = min(_cpu_count, 16 if _zarr_is_s3 else 32)
else:
    NUM_WORKERS = min(_cpu_count, 8)
PIN_MEMORY = True
USE_MULTI_GPU = False  # single GPU only
DEVICE = "cuda:7"

# --- generator (architecture) ---
GENERATOR = "gUnet"
# GENERATOR = "shalow_gUnet"
# GENERATOR = "shalow_gUnet_xs"

UNET_DIM = 32
UNET_DIM_MULTS = (1, 2, 4, 8)

# gUnet only — keep 0 for shalow variants
UNET_DROPOUT = 0.1
# UNET_DROPOUT = 0.2
# UNET_DROPOUT = 0.0

EPOCHS = 100
OPTIMIZER = "adamw"
WEIGHT_DECAY = 1e-3

BASE_RUN_TAG = RUN_TAG
RUN_TAG = weight_decay_run_tag(BASE_RUN_TAG, WEIGHT_DECAY)
RUN_TAG = dropout_run_tag(RUN_TAG, UNET_DROPOUT)
RUN_TAG = generator_run_tag(RUN_TAG, GENERATOR)
RUN_TAG = f"{RUN_TAG}_{EXPERIMENT_TAG}"
RUN_NAME = run_name(EXPORT_KEY, dt=DT, run_tag=RUN_TAG)
RUN_DIR = run_dir(MODELS_DIR, EXPORT_KEY, dt=DT, run_tag=RUN_TAG)

# --- loss spatial / coastal options (disabled) ---
# Coastal buffer: dilates land mask → less aggressive loss near coast. Do NOT use.
# LOSS_MASK_BUFFER_PX = 4
# Center-weighted loss: down-weights crop edges. Do NOT use for magic-window coastal.
# LOSS_SPATIAL_MODE = "gaussian_center"
# LOSS_GAUSSIAN_SIGMA = 32.0
LOSS_SPATIAL_MODE = "full"  # uniform L1/perceptual/adv over all ocean pixels (mask==0)

LOSS_MODE = "l1"  # L1 only (no perceptual / adversarial)
LOSS_METRIC = "components"
LOSS_GAUSSIAN_SIGMA = 32.0  # unused unless LOSS_SPATIAL_MODE == "gaussian_center"
CHANNEL_NAMES = ("u", "v")
iD = 2
BETA = 0.1
LAMBDA_PERC = 0.1
PERCEPTUAL_BACKEND = "vgg"
PERCEPTUAL_WEIGHTS = (0.1, 0.1, 1.0, 1.0, 1.0)
LPIPS_NET = "alex"
PERCEPTUAL_WARMUP_EPOCHS = 999  # unused when LOSS_MODE == "l1"
WARMUP_EPOCHS = 999
D_LR_RATIO = 0.5
D_SKIP_THRESHOLD = 0.2
USE_COSINE_LR = True
MERGE_VAL_EXPORT = False  # single pool export; split at training time
VAL_RECONSTRUCTION_METRIC = "l1"  # best checkpoint + val curves use masked uv L1
# VAL_RECONSTRUCTION_METRIC = "rmse"
# VAL_SPLIT_MODE = "export"  # use random export-time split from 01_export_magic_window.ipynb
DETACHED = True  # background subprocess — survives notebook disconnect
RESUME = False  # False = fresh run (overwrites RUN_DIR when PROTECT_EXISTING_RUN=False)
SHOW_INLINE_PLOTS = False
PREVIEW_LOSS_MASK = True
PREVIEW_VAL_MASK = True
PREVIEW_VAL_MASK_SAMPLES = 3
PROTECT_EXISTING_RUN = False
FILTER_FULLY_OCEAN_SAMPLES = False
VIZ_EVERY = 1
VIZ_SAMPLES = 1
VIZ_SEED = 42
VIZ_RESIDUAL_METRIC = "l1"

print("Working directory:", Path.cwd())
print("SHALOW_DIR:", SHALOW_DIR)
print("EXPORT_KEY:", EXPORT_KEY)
print("DATA_PREFIX:", DATA_PREFIX)
print("TEST_DATA_PREFIX:", TEST_DATA_PREFIX)
print("generator:", GENERATOR, "(unet_dim=", UNET_DIM, ", dropout=", UNET_DROPOUT, ")")
print("RUN_NAME:", RUN_NAME)
print("RUN_DIR:", RUN_DIR)
print("epochs:", EPOCHS)
print(f"Optimizer: {OPTIMIZER}, lr=2e-4, weight_decay={WEIGHT_DECAY}, epochs={EPOCHS}")
print("experiment tag:", EXPERIMENT_TAG)
print("dynamic random crop:", DYNAMIC_RANDOM_CROP)
print("train crops per anchor:", TRAIN_CROPS_PER_ANCHOR)
print(f"parallel: num_workers={NUM_WORKERS} (cpus={_cpu_count}) pin_memory={PIN_MEMORY} use_multi_gpu={USE_MULTI_GPU} device={DEVICE}")
print("detached:", DETACHED, "| resume:", RESUME)
print("val split:", f"{int((1 - VAL_SPLIT) * 100)}% train / {int(VAL_SPLIT * 100)}% val", f"({VAL_SPLIT_MODE}, seed={VAL_SPLIT_SEED})")
print("val metric:", VAL_RECONSTRUCTION_METRIC)
print("max_samples:", MAX_SAMPLES, "(all export sequences)")
print("rollout eval:", SHALOW_DIR / "05_Sh_evaluate_magic_window_rollout.ipynb")

Working directory: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow
SHALOW_DIR: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow
EXPORT_KEY: random_win_128
DATA_PREFIX: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/data/random_win_128/sample_13_22_uv_dt360
TEST_DATA_PREFIX: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/data/random_win_128/sample_23_uv_dt360
generator: gUnet (unet_dim= 32 , dropout= 0.1 )
RUN_NAME: shalow_adv_no_uv_13_22_random_win_128_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_dynwin_l1random_k1
RUN_DIR: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/Models/adv_no/shalow_adv_no_uv_13_22_random_win_128_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_dynwin_l1random_k1
epochs: 100
Optimizer: adamw, lr=2e-4, weight_decay=0.001, epochs=100
experiment tag: dynwin_l1random_k1
dynamic random crop: True
train crops per anchor: 1
parallel: num_workers=32 (cpus=96) pin_memory=True use_multi_gpu=True device=cuda:0
detached: True | resume: False
val split: 70% tr

In [2]:
assert "DATA_PREFIX" in globals(), "Run the setup cell above first."
print("Ready:", DATA_PREFIX)
print("Next: export check → train config → preview → train")


Ready: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/data/random_win_128/sample_13_22_uv_dt360
Next: export check → train config → preview → train


In [3]:
import json
import numpy as np

if "DATA_PREFIX" not in globals():
    raise RuntimeError("Run the setup cell above first.")

sims_path = Path(f"{DATA_PREFIX}_sims.npy")
if not sims_path.exists():
    raise FileNotFoundError(
        f"Missing pool export: {sims_path}\n"
        "Run Shalow/01_export_random_window.ipynb first."
    )

pool_sims = np.load(f"{DATA_PREFIX}_sims.npy", mmap_mode="r")
mask = np.load(f"{DATA_PREFIX}_mask.npy", mmap_mode="r")
window = np.load(f"{DATA_PREFIX}_window.npy")
windows_path = Path(f"{DATA_PREFIX}_windows.npy")
land_frac = float(mask.mean())
mask_frac_per = mask.mean(axis=(1, 2))
static_mask = len(mask) <= 1 or np.all(mask == mask[0:1])
n_use = len(pool_sims) if MAX_SAMPLES is None else min(int(MAX_SAMPLES), len(pool_sims))
n_train_split = int(n_use * (1.0 - VAL_SPLIT))
n_val_split = n_use - n_train_split

print(f"Pool export: {DATA_PREFIX}")
print(f"  sims shape: {pool_sims.shape}  (N, T, C, H, W)")
print(f"  sequences in pool: {n_use} / {len(pool_sims)}")
print(f"  mask: {land_frac:.2%} mean land | per-sample masks={not static_mask}")
print(f"  per-sample land: max {mask_frac_per.max():.4f} | ≥5%: {int((mask_frac_per >= 0.05).sum())}")
cfg_path = Path(f"{DATA_PREFIX}_export_config.json")
if cfg_path.exists():
    cfg = json.loads(cfg_path.read_text())
    print(f"  export max_mask_frac: {cfg.get('max_mask_frac')}")
print(f"  window sentinel: {window}  (per-sample coords in _windows.npy)")
if windows_path.exists():
    w = np.load(windows_path, mmap_mode="r")
    print(f"  windows.npy: {w.shape} [top, left, h, w] per sample")
print(
    f"  training split: {n_train_split} train + {n_val_split} val "
    f"({VAL_SPLIT_MODE}, {VAL_SPLIT:.0%} val, seed={VAL_SPLIT_SEED})"
)
if mask_frac_per.max() >= 0.05:
    raise ValueError(
        f"Export has samples with ≥5% land (max {mask_frac_per.max():.4f}). "
        "Re-run Shalow/01_export_random_window.ipynb with MAX_MASK_FRAC=0.05."
    )


Pool export: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/data/random_win_128/sample_13_22_uv_dt360
  sims shape: (7304, 4, 2, 128, 128)  (N, T, C, H, W)
  sequences in pool: 7304 / 7304
  mask: 0.38% mean land | per-sample masks=True
  per-sample land: max 0.0500 | ≥5%: 0
  export max_mask_frac: 0.05
  window sentinel: [ -1.  -1. 128. 128.   0.]  (per-sample coords in _windows.npy)
  windows.npy: (7304, 4) [top, left, h, w] per sample
  training split: 5112 train + 2192 val (random, 30% val, seed=42)


In [4]:
# Collect and save dataset statistics for cross-dataset comparison
assert "EXPORT_KEY" in globals(), "Run the setup cell first."

RUN_DIR.mkdir(parents=True, exist_ok=True)
dataset_stats = collect_dataset_stats(DATA_DIR, EXPORT_KEY, dt=DT)
stats_path = save_dataset_stats(RUN_DIR, dataset_stats)
index_path = update_comparison_index(
    MODELS_DIR,
    run_name_value=RUN_NAME,
    export_key=EXPORT_KEY,
    run_dir_path=RUN_DIR,
    dataset_stats=dataset_stats,
)

print(f"Saved dataset stats: {stats_path}")
print(
    f"  train {dataset_stats['train']['n_sequences']} | "
    f"val {dataset_stats['val']['n_sequences']} | "
    f"test {dataset_stats['test']['n_sequences']} | "
    f"land {dataset_stats['train']['land_fraction']:.2%}"
)
print(f"Updated comparison index: {index_path}")


Saved dataset stats: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/Models/adv_no/shalow_adv_no_uv_13_22_random_win_128_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_dynwin_l1random_k1/dataset_stats.json
  train 7304 | val 0 | test 319 | land 0.38%
Updated comparison index: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/Models/adv_no/comparison_index.json


In [5]:
train_config = AdvNOTrainConfig(
    data_prefix=DATA_PREFIX,
    run_name=RUN_NAME,
    dt_hours=DT,
    epochs=EPOCHS,
    batch_size=64,
    lr=2e-4,
    optimizer=OPTIMIZER,
    weight_decay=WEIGHT_DECAY,
    d_lr=1e-4,
    d_lr_follow_g=True,
    d_lr_ratio=D_LR_RATIO,
    d_skip_threshold=D_SKIP_THRESHOLD,
    use_cosine_lr=USE_COSINE_LR,
    iD=iD,
    generator=GENERATOR,
    unet_dim=UNET_DIM,
    unet_dim_mults=UNET_DIM_MULTS,
    unet_dropout=UNET_DROPOUT,
    device=DEVICE,
    loss_mode=LOSS_MODE,
    loss_metric=LOSS_METRIC,
    loss_spatial_mode=LOSS_SPATIAL_MODE,
    loss_gaussian_sigma=LOSS_GAUSSIAN_SIGMA,
    loss_mask_buffer_px=0,
    filter_fully_ocean_samples=FILTER_FULLY_OCEAN_SAMPLES,
    max_samples=MAX_SAMPLES,
    protect_existing_run=PROTECT_EXISTING_RUN,
    preview_loss_mask=PREVIEW_LOSS_MASK,
    preview_val_mask=PREVIEW_VAL_MASK,
    beta=BETA,
    lambda_perc=LAMBDA_PERC,
    perceptual_backend=PERCEPTUAL_BACKEND,
    lpips_net=LPIPS_NET,
    perceptual_weights=PERCEPTUAL_WEIGHTS,
    perceptual_warmup_epochs=PERCEPTUAL_WARMUP_EPOCHS,
    warmup_epochs=WARMUP_EPOCHS,
    checkpoint_subdir=str(MODELS_DIR),
    isolate_runs=True,
    channel_names=CHANNEL_NAMES,
    viz_every=VIZ_EVERY,
    viz_samples=VIZ_SAMPLES,
    viz_seed=VIZ_SEED,
    viz_residual_metric=VIZ_RESIDUAL_METRIC,
    show_inline_plots=SHOW_INLINE_PLOTS,
    val_split=VAL_SPLIT,
    val_split_mode=VAL_SPLIT_MODE,
    val_split_seed=VAL_SPLIT_SEED,
    val_reconstruction_metric=VAL_RECONSTRUCTION_METRIC,
    val_data_prefix=None,
    merge_val_export_prefix=None,
    dynamic_random_crop=DYNAMIC_RANDOM_CROP,
    random_crop_seed=RANDOM_CROP_SEED,
    max_mask_frac=MAX_MASK_FRAC,
    max_crop_attempts=200,
    train_crops_per_anchor=TRAIN_CROPS_PER_ANCHOR,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    use_multi_gpu=USE_MULTI_GPU,
)

(_data, _mask, _day, _hour, _, _), _val_tensors = load_notebook_training_arrays(train_config)
_n_total = len(_data)
if train_config.val_data_prefix:
    _n_val = len(_val_tensors[0])
else:
    _n_val = None
_train_idx, _val_idx = resolve_train_val_indices(
    train_config, _n_total, n_val=_n_val
)

RUN_DIR.mkdir(parents=True, exist_ok=True)
config_path = RUN_DIR / f"train_config_{RUN_NAME}.json"
config_path.write_text(json.dumps(asdict(train_config), indent=2, default=str))
print(f"Run directory: {RUN_DIR}")
print(f"Wrote config:  {config_path}")
print(f"  data: {DATA_PREFIX} | iD={iD} | loss_metric={LOSS_METRIC}")
split_note = f"{VAL_SPLIT_MODE} split on pool export ({VAL_SPLIT:.0%} val, seed={VAL_SPLIT_SEED}): {DATA_PREFIX}"
print(
    f"  sequences: {_n_total} total | "
    f"train {len(_train_idx)} | val {len(_val_idx)} ({split_note})"
)
if LOSS_MODE == "l1":
    print("  loss mode: L1-only (no perceptual / discriminator)")
elif LOSS_MODE == "adv_no":
    perc_name = f"LPIPS ({LPIPS_NET})" if PERCEPTUAL_BACKEND == "lpips" else "VGG-19"
    if PERCEPTUAL_WARMUP_EPOCHS == 0 and WARMUP_EPOCHS == 0:
        print(f"  loss mode: adv-NO — L1 + {perc_name} + adversarial from epoch 1")
    else:
        print(
            f"  loss mode: adv-NO — L1 through ep {PERCEPTUAL_WARMUP_EPOCHS}, "
            f"+{perc_name} from ep {PERCEPTUAL_WARMUP_EPOCHS + 1}, "
            f"+adv from ep {WARMUP_EPOCHS + 1}"
        )
else:
    print(f"  loss mode: {LOSS_MODE!r}")
if LOSS_SPATIAL_MODE == "full":
    spatial_note = "uniform over whole valid field"
elif LOSS_SPATIAL_MODE == "gaussian_center":
    spatial_note = f"gaussian_center (σ={LOSS_GAUSSIAN_SIGMA}) — state⊙G"
else:
    spatial_note = LOSS_SPATIAL_MODE
print(f"  loss spatial: {spatial_note}")
print(f"  rollout: best_model.pth + norm_stats.pt → notebook 05")


Train crops per anchor: using 3652 / 7304 export rows (crop slots < 1)
Run directory: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/Models/adv_no/shalow_adv_no_uv_13_22_random_win_128_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_dynwin_l1random_k1
Wrote config:  /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/Models/adv_no/shalow_adv_no_uv_13_22_random_win_128_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_dynwin_l1random_k1/train_config_shalow_adv_no_uv_13_22_random_win_128_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_dynwin_l1random_k1.json
  data: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/data/random_win_128/sample_13_22_uv_dt360 | iD=2 | loss_metric=components
  sequences: 3652 total | train 2557 | val 1095 (random split on pool export (30% val, seed=42): /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/data/random_win_128/sample_13_22_uv_dt360)
  loss mode: L1-only (no perceptual / discriminator)
  loss spatial: uniform over whole valid field
  rollout: best_model.pth + no

In [6]:
import numpy as np
from train.adv_no_train import _run_has_artifacts
from src.train.fix_window_train import SimulationDataset
from src.train.fix_window_viz import (
    fixed_viz_indices,
    save_loss_mask_preview,
    save_validation_mask_preview,
)

assert "train_config" in globals(), "Run the train config cell above first."
_mask_preview = np.load(f"{DATA_PREFIX}_mask.npy", mmap_mode="r")
land_frac = float(_mask_preview.mean())

if PROTECT_EXISTING_RUN and _run_has_artifacts(RUN_DIR, RUN_NAME):
    raise FileExistsError(
        f"Run already exists: {RUN_DIR}\n"
        "Change RUN_TAG to a new name before preview / training."
    )
elif _run_has_artifacts(RUN_DIR, RUN_NAME):
    print(f"Note: {RUN_DIR} exists — preview/training will overwrite prior results.")

if PREVIEW_LOSS_MASK:
    preview_path = save_loss_mask_preview(
        DATA_PREFIX,
        RUN_DIR / "viz" / "loss_mask_preview.png",
        sample_index=0,
        loss_spatial_mode=LOSS_SPATIAL_MODE,
        loss_gaussian_sigma=LOSS_GAUSSIAN_SIGMA,
        loss_mask_buffer_px=0,
        channel_names=CHANNEL_NAMES,
        show=SHOW_INLINE_PLOTS,
    )
    print(f"Loss mask preview: {preview_path}")
else:
    print("Loss mask preview skipped (PREVIEW_LOSS_MASK=False)")

if PREVIEW_VAL_MASK:
    (_data, _mask, _day, _hour, _, _), _val_tensors = load_notebook_training_arrays(train_config)
    if train_config.val_data_prefix:
        _val_data, _val_mask, _val_day, _val_hour = _val_tensors[:4]
    else:
        _val_data = _val_mask = _val_day = _val_hour = None
    (
        _tr_data,
        _tr_mask,
        _tr_day,
        _tr_hour,
        _va_data,
        _va_mask,
        _va_day,
        _va_hour,
        _train_idx,
        _val_idx,
    ) = prepare_train_val_tensors(
        _data,
        _mask,
        _day,
        _hour,
        train_config,
        val_data=_val_data,
        val_mask=_val_mask,
        val_day=_val_day,
        val_hour=_val_hour,
    )
    _train_set = SimulationDataset(_tr_data, _tr_mask, _tr_day, _tr_hour, iD=iD)
    _val_set = SimulationDataset(
        _va_data, _va_mask, _va_day, _va_hour,
        mean=_train_set.mean, std=_train_set.std, iD=iD,
    )
    _viz_idx = fixed_viz_indices(PREVIEW_VAL_MASK_SAMPLES, len(_val_set), VIZ_SEED)
    _global_viz_idx = [int(_val_idx[i]) for i in _viz_idx]
    val_mask_path = save_validation_mask_preview(
        _val_set,
        RUN_DIR / "viz" / "val_mask_preview.png",
        sample_indices=_viz_idx,
        global_sample_indices=_global_viz_idx,
        channel_names=CHANNEL_NAMES,
        loss_mask_buffer_px=0,
        show=SHOW_INLINE_PLOTS,
    )
    print(f"Val mask preview: {val_mask_path}")
else:
    print("Val mask preview skipped (PREVIEW_VAL_MASK=False)")


Loss mask preview: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/Models/adv_no/shalow_adv_no_uv_13_22_random_win_128_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_dynwin_l1random_k1/viz/loss_mask_preview.png
Train crops per anchor: using 3652 / 7304 export rows (crop slots < 1)
Train/val split (random, seed=42): 2557 train sequences, 1095 val sequences (30% val, 3652 total after mask filter)
Val mask preview: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/Models/adv_no/shalow_adv_no_uv_13_22_random_win_128_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_dynwin_l1random_k1/viz/val_mask_preview.png


## Train

**`DETACHED = True`** (default): launches `scripts/train_shalow_adv_no.py` in a background process on the **remote server** — keeps running if you close the laptop or lose Cursor connection.

- Monitor: `tail -f Shalow/logs/train_{RUN_NAME}.log`
- After interrupt / reboot: set **`RESUME = True`**, re-run setup + train-config + this cell
- **Fresh run:** `RESUME = False`, `PROTECT_EXISTING_RUN = False` — new or overwrite `{RUN_DIR}`
- Foreground: set `DETACHED = False` (stops when notebook kernel stops)


In [7]:
LOG_DIR.mkdir(parents=True, exist_ok=True)
log_path = LOG_DIR / f"train_{RUN_NAME}.log"

if DETACHED:
    cmd = [
        sys.executable,
        str(ROOT / "scripts/train_shalow_adv_no.py"),
        "--config",
        str(config_path),
    ]
    if RESUME:
        cmd.append("--resume")
    log_mode = "a" if RESUME and log_path.exists() else "w"
    with open(log_path, log_mode) as log_file:
        proc = subprocess.Popen(
            cmd,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            start_new_session=True,
            cwd=ROOT,
        )
    print(f"Started background training PID {proc.pid}")
    print(f"Log: {log_path} ({log_mode})")
    print(f"Monitor: tail -f {log_path}")
    if RESUME:
        print("Resume mode — continuing from last completed epoch if checkpoint exists")
    print(
        f"Resume after disconnect: set RESUME=True and re-run, or:\n"
        f"  python scripts/train_shalow_adv_no.py --config {config_path} --resume"
    )
    print("Stop with: kill", proc.pid)
else:
    class _Tee:
        def __init__(self, *streams):
            self.streams = streams

        def write(self, data):
            for stream in self.streams:
                stream.write(data)
                stream.flush()

        def flush(self):
            for stream in self.streams:
                stream.flush()

    print(f"Foreground training — log: {log_path}")
    print("Stop anytime: Kernel → Interrupt (or ■ button)\n")

    _stdout = sys.stdout
    with open(log_path, "w") as log_file:
        sys.stdout = _Tee(_stdout, log_file)
        try:
            run_adv_no_training(train_config, resume=RESUME)
        except KeyboardInterrupt:
            print("\nNotebook interrupt received — training stopped.")
        finally:
            sys.stdout = _stdout

    print(f"\nLog saved: {log_path}")


Started background training PID 1171063
Log: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/logs/train_shalow_adv_no_uv_13_22_random_win_128_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_dynwin_l1random_k1.log (w)
Monitor: tail -f /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/logs/train_shalow_adv_no_uv_13_22_random_win_128_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_dynwin_l1random_k1.log
Resume after disconnect: set RESUME=True and re-run, or:
  python scripts/train_shalow_adv_no.py --config /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/Models/adv_no/shalow_adv_no_uv_13_22_random_win_128_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_dynwin_l1random_k1/train_config_shalow_adv_no_uv_13_22_random_win_128_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_dynwin_l1random_k1.json --resume
Stop with: kill 1171063


## Loss curves (after epoch 1+)

In [8]:
import matplotlib.pyplot as plt
import torch

history_path = RUN_DIR / "history" / f"training_history_{RUN_NAME}.pt"
state_path = RUN_DIR / f"train_state_{RUN_NAME}.json"
plot_path = RUN_DIR / "viz" / f"training_loss_curve_{RUN_NAME}.png"

if not history_path.exists():
    msg = [f"No history yet: {history_path}"]
    if state_path.exists():
        state = json.loads(state_path.read_text())
        msg.append(
            f"Training status: {state.get('status')} | "
            f"last completed epoch: {state.get('last_completed_epoch', 0)}/{state.get('epochs_planned', '?')}"
        )
    if log_path.exists():
        tail = log_path.read_text().splitlines()[-5:]
        msg.append("Recent log lines:")
        msg.extend(f"  {line}" for line in tail)
    msg.append("Re-run this cell after epoch 1 finishes.")
    print("\n".join(msg))
else:
    hist = torch.load(history_path, map_location="cpu", weights_only=False)
    epochs = hist["epoch"]

    if not epochs:
        print("History file exists but is empty. Wait for epoch 1 to finish, then re-run.")
    else:
        recon_label = "speed L1" if LOSS_METRIC == "speed" else "uv L1"
        fig, ax = plt.subplots(1, 3, figsize=(14, 4))
        ax[0].plot(epochs, hist["train_l1"], label=f"Train {recon_label}")
        ax[0].plot(epochs, hist["val_l1"], label=f"Val {recon_label}")
        ax[0].set_title(f"Reconstruction ({recon_label})")
        ax[0].legend()
        ax[0].grid(True, alpha=0.3)

        use_adv_no = LOSS_MODE == "adv_no"
        perc_warmup = PERCEPTUAL_WARMUP_EPOCHS if use_adv_no else 0
        adv_warmup = WARMUP_EPOCHS if use_adv_no else 0
        perc_title = f"LPIPS ({LPIPS_NET})" if PERCEPTUAL_BACKEND == "lpips" else "VGG-19"

        if use_adv_no and any(hist.get("train_perceptual", [])):
            ax[1].plot(epochs, hist["train_perceptual"], label="Train perc")
            ax[1].plot(epochs, hist["val_perceptual"], label="Val perc")
            if perc_warmup > 0:
                ax[1].axvline(perc_warmup + 0.5, color="#2ca02c", linestyle=":", alpha=0.7)
            ax[1].set_title(f"Perceptual ({perc_title})")
            ax[1].legend()
            ax[1].grid(True, alpha=0.3)
        else:
            ax[1].set_title("Perceptual (starts after warmup)")
            ax[1].axis("off")

        if use_adv_no and any(hist.get("train_adv", [])):
            ax[2].plot(epochs, hist["train_adv"], label="Train adv")
            ax[2].plot(epochs, hist["train_d"], label="Train D")
            if adv_warmup > perc_warmup:
                ax[2].axvline(adv_warmup + 0.5, color="#888888", linestyle=":", alpha=0.7)
            ax[2].set_title("Adversarial terms")
            ax[2].legend()
            ax[2].grid(True, alpha=0.3)
        else:
            ax[2].set_title("Adversarial (starts after warmup)")
            ax[2].axis("off")

        plt.tight_layout()
        plot_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(plot_path, dpi=150, bbox_inches="tight")
        if SHOW_INLINE_PLOTS:
            plt.show()
        else:
            plt.close(fig)
        print(f"Best val {recon_label}: {hist['best_val_loss']:.6f} at epoch {hist['best_epoch']}")
        print(f"Saved plot: {plot_path}")
        update_comparison_index(
            MODELS_DIR,
            run_name_value=RUN_NAME,
            export_key=EXPORT_KEY,
            run_dir_path=RUN_DIR,
            dataset_stats=dataset_stats if "dataset_stats" in globals() else None,
            training_summary={
                "best_val_loss": float(hist["best_val_loss"]),
                "best_epoch": int(hist["best_epoch"]),
                "epochs_completed": int(epochs[-1]) if epochs else 0,
            },
        )

No history yet: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/Models/adv_no/shalow_adv_no_uv_13_22_random_win_128_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_dynwin_l1random_k1/history/training_history_shalow_adv_no_uv_13_22_random_win_128_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_dynwin_l1random_k1.pt
Recent log lines:
Re-run this cell after epoch 1 finishes.


## Next: rollout evaluation

After training completes, open [05_evaluate_adv_no_magic_window_rollout.ipynb](05_evaluate_adv_no_magic_window_rollout.ipynb).

It loads `{RUN_DIR}/best_model.pth` and saves rollout metrics to `Shalow/Test&Rollout/{RUN_NAME}/rollout_results.json`. Set the same `EXPORT_KEY` in notebook 05.

In [9]:
print(f"Model dir: {RUN_DIR}")
print(f"  best_model: {RUN_DIR / 'best_model.pth'}")
print(f"  norm_stats: {RUN_DIR / 'norm_stats.pt'}")
print("Open 05_evaluate_adv_no_magic_window_rollout.ipynb when training is done.")

Model dir: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/Models/adv_no/shalow_adv_no_uv_13_22_random_win_128_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_dynwin_l1random_k1
  best_model: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/Models/adv_no/shalow_adv_no_uv_13_22_random_win_128_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_dynwin_l1random_k1/best_model.pth
  norm_stats: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/Models/adv_no/shalow_adv_no_uv_13_22_random_win_128_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_dynwin_l1random_k1/norm_stats.pt
Open 05_evaluate_adv_no_magic_window_rollout.ipynb when training is done.
